In [1]:
import numpy as np
import pandas as pd
import hypertools as hyp
import brainiak.eventseg.event as event
import pickle
from scipy.spatial.distance import cdist
import matplotlib.pyplot as plt
import seaborn as sns

%matplotlib inline

In [2]:
annot_dir = '../../../data/annotations_dfs/'
model_dir = '../../../data/models/video/t100_w50/'
transc_dir = '../../../data/transcriptions/automatic/'
pickle_dir = '../../../data/pickles/'

In [3]:
atlep1_model = np.load(model_dir+'/atlep1_model_t100_w50_res.npy')
atlep2_model = np.load(model_dir+'/atlep2_model_t100_w50_res.npy')
arrdev_model = np.load(model_dir+'/arrdev_model_t100_w50_res.npy')
ep_models_dict = {'atlep1': atlep1_model, 'atlep2' : atlep2_model, 'arrdev' : arrdev_model}

In [4]:
with open(pickle_dir+'/participant_models.p', 'rb') as f:
    participant_models = pickle.load(f)

In [5]:
def reduce_model(m, ev):
    """
    Reduce a model based on event labels
    """
    w = (np.round(ev.segments_[0])==1).astype(bool)
    return np.array([m[wi, :].mean(0) for wi in w.T])

## find optimal k for each recall model

In [6]:
ks = list(range(2,30))
# participant_max_ks = {ep : [(s[0]) for s in subj] for subj in participant_models[ep]
#                       for (ep, subj) in participant_models.items()}
participant_max_ks = {'atlep1' : [], 'atlep2' : [], 'arrdev' : [], 'prediction' : [],
                     'delayed' : []}

for a_type, participants in participant_models.items():
    if a_type == 'arrdev':
        for participant in participants[len(participants)//2:]:
            turkid, model = participant[0], participant[1]
            print('fitting ' + turkid + '...')
            mcorr = np.corrcoef(model)
            cs = []
            for k in ks:
                print('\tk = ' + str(k))
                ev = event.EventSegment(k)
                ev.fit(model)
                i1, i2 = np.where(np.round(ev.segments_[0])==1)
                w = np.zeros_like(ev.segments_[0])
                w[i1,i2] = 1
                w = np.dot(w, w.T).astype(bool)
                c = mcorr[w].mean()/mcorr[~w].mean() - k/1000
                cs.append(c)
            m = ks[np.argmax(cs)]
            participant_max_ks[a_type].append((turkid, m))

fitting debug8bX3q:debugdsqKI...
	k = 2
	k = 3
	k = 4
	k = 5
	k = 6
	k = 7
	k = 8
	k = 9
	k = 10
	k = 11
	k = 12
	k = 13
	k = 14
	k = 15
	k = 16
	k = 17
	k = 18
	k = 19
	k = 20
	k = 21
	k = 22
	k = 23
	k = 24
	k = 25
	k = 26
	k = 27
	k = 28
	k = 29
fitting debugAlono:debugbY9Uu...
	k = 2
	k = 3
	k = 4
	k = 5
	k = 6
	k = 7
	k = 8
	k = 9
	k = 10
	k = 11
	k = 12
	k = 13
	k = 14
	k = 15
	k = 16
	k = 17
	k = 18
	k = 19
	k = 20
	k = 21
	k = 22
	k = 23
	k = 24
	k = 25
	k = 26
	k = 27
	k = 28
	k = 29
fitting debugjVH6O:debugDbg24...
	k = 2
	k = 3
	k = 4
	k = 5
	k = 6
	k = 7
	k = 8
	k = 9
	k = 10
	k = 11
	k = 12
	k = 13
	k = 14
	k = 15
	k = 16
	k = 17
	k = 18
	k = 19
	k = 20
	k = 21
	k = 22
	k = 23
	k = 24
	k = 25
	k = 26
	k = 27
	k = 28
	k = 29
fitting debug27k0b:debugOGsyi...
	k = 2
	k = 3
	k = 4
	k = 5
	k = 6
	k = 7
	k = 8
	k = 9
	k = 10
	k = 11
	k = 12
	k = 13
	k = 14
	k = 15
	k = 16
	k = 17
	k = 18
	k = 19
	k = 20
	k = 21
	k = 22
	k = 23
	k = 24
	k = 25
	k = 26
	k = 27
	k = 28
	k = 29
fitt

In [7]:
arrdev_max_ks = participant_max_ks['arrdev']

with open(pickle_dir+'/eventseg-models/tmp-parallel-output/arrdev_max_ks-1.p', 'wb') as f:
    pickle.dump(arrdev_max_ks, f)

## fit eventseg model to recall using optimal k